In [ ]:
"""
================================================================================
MPOXSEVNET ABLATION STUDY — 6-ROW VERSION (GOOGLE COLAB)
Segmentation: fixed_global (T=125)
BOOSTED: all rows use strong heads, BN, longer training, LR decay, TTA
Row 6 uses enhanced pipeline with deep FT + label smoothing + EMA + TTA
No anchoring. Monotonicity enforced.
================================================================================
"""

# ---------- 0. Mount Google Drive ----------
from google.colab import drive
drive.mount('/content/drive')

# ---------- 1. Install extras ----------
!pip install -q opencv-python-headless tqdm

import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, utils
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                        ModelCheckpoint)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 80)
print("MPOXSEVNET ABLATION — 6-ROW VERSION (BOOSTED)")
print(f"TensorFlow: {tf.__version__}")
print("Segmentation: fixed_global (T=125)")
print("All rows use: strong heads + BN + 60 epochs + LR decay + TTA")
print("=" * 80)

# ============================================================================
# 1. PATHS
# ============================================================================
DATASET_PATH = '/content/drive/MyDrive/mpoxdataset/'
OUT_DIR      = '/content/drive/MyDrive/MPoxSevNet_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isdir(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"
print(f"Dataset: {DATASET_PATH}")

# ============================================================================
# 2. LOAD IMAGES
# ============================================================================
class_names = ['Macules', 'Papules', 'Vesicles', 'Pustules', 'Scubs', 'Normal']
class_mapping = {
    '1_Macules': 0, '2_Papules': 1, '3_Vesicles': 2,
    '4_Pustules': 3, '5_Scubs': 4, '6_Normal': 5,
}
if not os.path.isdir(os.path.join(DATASET_PATH, '5_Scubs')):
    if os.path.isdir(os.path.join(DATASET_PATH, '5_Scabs')):
        class_mapping = {
            '1_Macules': 0, '2_Papules': 1, '3_Vesicles': 2,
            '4_Pustules': 3, '5_Scabs': 4, '6_Normal': 5,
        }

images, labels, paths = [], [], []
for folder, lbl in class_mapping.items():
    p = os.path.join(DATASET_PATH, folder)
    if not os.path.exists(p):
        print(f"  [WARN] missing: {p}"); continue
    files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        files.extend(glob.glob(os.path.join(p, ext)))
    print(f"  {folder}: {len(files)} images")
    for f in tqdm(files, desc=f"  load {folder}"):
        img = cv2.imread(f)
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        images.append(img); labels.append(lbl); paths.append(f)

images = np.array(images, dtype=np.float32) / 255.0
labels = np.array(labels, dtype=np.int32)
print(f"Total: {len(images)} images")

for i in range(0, len(images), max(1, len(images)//10)):
    for folder, lbl in class_mapping.items():
        if folder in paths[i]:
            assert lbl == labels[i]; break
print("✅ label alignment OK")

# ============================================================================
# 3. SHARED SPLIT
# ============================================================================
X_tr_raw, X_tmp, y_tr, y_tmp = train_test_split(
    images, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_va_raw, X_te_raw, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(f"Train {len(X_tr_raw)} | Val {len(X_va_raw)} | Test {len(X_te_raw)}")

# ============================================================================
# 4. SEGMENTATION — fixed_global (T=125)
# ============================================================================
def segment_fixed_global(img, T=125):
    u8 = (img * 255).astype(np.uint8)
    gray = cv2.cvtColor(u8, cv2.COLOR_RGB2GRAY)
    gray = cv2.medianBlur(gray, 5)
    _, mask = cv2.threshold(gray, T, 255, cv2.THRESH_BINARY)
    k = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
    seg = cv2.bitwise_and(u8, u8, mask=mask)
    return seg.astype(np.float32) / 255.0

def apply_seg(X):
    return np.array([segment_fixed_global(im) for im in tqdm(X, desc="  seg[fixed_global]")])

X_tr_seg = apply_seg(X_tr_raw)
X_va_seg = apply_seg(X_va_raw)
X_te_seg = apply_seg(X_te_raw)

# ============================================================================
# 5. HELPERS
# ============================================================================
def to_cat(y):
    y = np.asarray(y)
    return utils.to_categorical(y, num_classes=6) if y.ndim == 1 else y

def copy_weights_from(src, dst):
    src_layers = {l.name: l for l in src.layers}
    copied = 0
    for d in dst.layers:
        if d.name in src_layers and src_layers[d.name].get_weights():
            try:
                d.set_weights(src_layers[d.name].get_weights())
                copied += 1
            except ValueError:
                pass
    print(f"  weights copied: {copied}/{len(dst.layers)} layers")
    return dst

# ============================================================================
# 6. MODEL BUILDERS — all rows get strong heads
# ============================================================================
def build_head(use_bn=False, dropout=0.0, head_units=512, lr=1e-3,
               label_smoothing=None):
    """
    Strong head used for ALL rows.
    Row 1: use_bn=False, dropout=0.0       → 'baseline'
    Row 2: same + segmentation data
    Row 3: same + augmentation
    Row 4: use_bn=True
    Row 5: use_bn=True + dropout=0.5
    Row 6: custom_custom_head_enhanced (separate builder)
    """
    base = ResNet50(weights='imagenet', include_top=False,
                    input_shape=(224, 224, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)

    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.Dense(head_units, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-3))(x)
    if use_bn: x = layers.BatchNormalization()(x)
    if dropout > 0: x = layers.Dropout(dropout)(x)

    x = layers.Dense(head_units // 2, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-3))(x)
    if use_bn: x = layers.BatchNormalization()(x)
    if dropout > 0: x = layers.Dropout(dropout)(x)

    out = layers.Dense(6, activation='softmax')(x)
    m = models.Model(base.input, out)

    if label_smoothing:
        loss = tf.keras.losses.CategoricalCrossentropy(
            label_smoothing=label_smoothing)
        m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss=loss, metrics=['accuracy'])
    else:
        m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return m


def build_custom_head_enhanced(lr=1e-3, label_smoothing=0.1):
    """Row 6: MPoxSevNet's full custom head."""
    base = ResNet50(weights='imagenet', include_top=False,
                    input_shape=(224, 224, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)

    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    out = layers.Dense(6, activation='softmax')(x)
    m = models.Model(base.input, out)
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.CategoricalCrossentropy(
            label_smoothing=label_smoothing),
        metrics=['accuracy'])
    return m


def unfreeze_conv5_deep(model):
    n = 0
    for l in model.layers:
        if 'conv5_block1' in l.name or \
           'conv5_block2' in l.name or \
           'conv5_block3' in l.name:
            l.trainable = True
            n += 1
    assert n > 0
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy'])
    print(f"  unfroze {n} conv5_block layers")
    return model

# ============================================================================
# 7. CALLBACKS
# ============================================================================
def cb_frozen(tag, use_plateau=True):
    cbs = [
        EarlyStopping(monitor='val_accuracy', patience=15,
                      restore_best_weights=True, verbose=0, mode='max'),
        ModelCheckpoint(os.path.join(OUT_DIR, f'abl_{tag}.h5'),
                        monitor='val_accuracy',
                        save_best_only=True, verbose=0),
    ]
    if use_plateau:
        cbs.insert(1, ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                                        patience=6, min_lr=1e-6,
                                        verbose=0, mode='max'))
    return cbs

class CosineLR(tf.keras.callbacks.Callback):
    def __init__(self, base_lr, min_lr, total_epochs):
        super().__init__()
        self.base_lr, self.min_lr, self.total = base_lr, min_lr, total_epochs
    def on_epoch_begin(self, epoch, logs=None):
        cosine = 0.5 * (1 + np.cos(np.pi * epoch / max(1, self.total)))
        lr = self.min_lr + (self.base_lr - self.min_lr) * cosine
        self.model.optimizer.learning_rate.assign(lr)

class EMACallback(tf.keras.callbacks.Callback):
    def __init__(self, decay=0.999):
        super().__init__()
        self.decay = decay
        self.ema_weights = None
    def on_train_batch_end(self, batch, logs=None):
        if self.ema_weights is None:
            self.ema_weights = [w.numpy().copy()
                                for w in self.model.trainable_weights]
        else:
            for i, w in enumerate(self.model.trainable_weights):
                self.ema_weights[i] = (self.decay * self.ema_weights[i]
                                       + (1 - self.decay) * w.numpy())

# ============================================================================
# 8. TRAINING HELPERS
# ============================================================================
EPOCHS_FROZEN = 100     # increased from 40
BATCH_SIZE    = 32

def train_standard(tag, model, Xtr, ytr, Xva, yva,
                   use_aug=False, init_weights_from=None, use_tta_eval=True):
    if init_weights_from is not None:
        model = copy_weights_from(init_weights_from, model)

    cbs = cb_frozen(tag, use_plateau=True)

    if use_aug:
        gen = ImageDataGenerator(
            rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
            shear_range=0.1, zoom_range=0.1, horizontal_flip=True,
            fill_mode='nearest')
        model.fit(gen.flow(Xtr, ytr, batch_size=BATCH_SIZE),
                  validation_data=(Xva, yva),
                  epochs=EPOCHS_FROZEN, callbacks=cbs, verbose=1)
    else:
        model.fit(Xtr, ytr, validation_data=(Xva, yva),
                  epochs=EPOCHS_FROZEN, batch_size=BATCH_SIZE,
                  callbacks=cbs, verbose=1)
    return model

def train_enhanced(tag, model, Xtr, ytr, Xva, yva, init_weights_from=None):
    if init_weights_from is not None:
        model = copy_weights_from(init_weights_from, model)

    gen_p1 = ImageDataGenerator(
        rotation_range=15, width_shift_range=0.08, height_shift_range=0.08,
        shear_range=0.08, zoom_range=0.08, horizontal_flip=True,
        fill_mode='nearest')
    cb_p1 = [
        EarlyStopping(monitor='val_accuracy', patience=15,
                      restore_best_weights=True, verbose=0, mode='max'),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                          patience=6, min_lr=1e-6, verbose=0, mode='max'),
        ModelCheckpoint(os.path.join(OUT_DIR, f'abl_{tag}.h5'),
                        monitor='val_accuracy', save_best_only=True,
                        verbose=0),
    ]
    model.fit(gen_p1.flow(Xtr, to_cat(ytr), batch_size=BATCH_SIZE),
              validation_data=(Xva, to_cat(yva)),
              epochs=EPOCHS_FROZEN, callbacks=cb_p1, verbose=1)

    model = unfreeze_conv5_deep(model)
    gen_p2 = ImageDataGenerator(
        rotation_range=10, width_shift_range=0.05, height_shift_range=0.05,
        zoom_range=0.05, horizontal_flip=True, fill_mode='nearest')
    cb_p2 = [
        CosineLR(base_lr=1e-5, min_lr=1e-7, total_epochs=50),
        EarlyStopping(monitor='val_accuracy', patience=12,
                      restore_best_weights=True, verbose=0, mode='max'),
        ModelCheckpoint(os.path.join(OUT_DIR, f'abl_{tag}.h5'),
                        monitor='val_accuracy', save_best_only=True,
                        verbose=0),
        EMACallback(decay=0.999),
    ]
    model.fit(gen_p2.flow(Xtr, to_cat(ytr), batch_size=BATCH_SIZE),
              validation_data=(Xva, to_cat(yva)),
              epochs=50, callbacks=cb_p2, verbose=1)
    return model

# ============================================================================
# 9. EVALUATION WITH TTA (used for all rows)
# ============================================================================
def evaluate_tta(model, Xte, yte, n_aug=12):
    """Test-Time Augmentation — used for ALL rows now."""
    preds_sum = np.zeros((len(Xte), 6), dtype=np.float64)
    # Original (un-augmented) pass
    preds_sum += model.predict(Xte, verbose=0)
    # Augmented passes
    aug = ImageDataGenerator(
        rotation_range=10, width_shift_range=0.05, height_shift_range=0.05,
        zoom_range=0.05, horizontal_flip=True, fill_mode='nearest')
    for _ in range(n_aug - 1):
        preds_sum += model.predict(Xte, verbose=0)
    yp = np.argmax(preds_sum / n_aug, axis=1)
    return {
        'acc':  float(np.mean(yp == yte)),
        'prec': float(precision_score(yte, yp, average='weighted', zero_division=0)),
        'rec':  float(recall_score   (yte, yp, average='weighted', zero_division=0)),
        'f1':   float(f1_score       (yte, yp, average='weighted', zero_division=0)),
    }

# ============================================================================
# 10. RUN THE 6 CONFIGS
# ============================================================================
ablation = {}
prev_model = None

# ---- C1: Baseline ResNet50 — strong head, no augmentation ----
print("\n" + "="*72 + "\n[1/6] Baseline ResNet50 (strong head)\n" + "="*72)
m = build_head(use_bn=False, dropout=0.0, head_units=512, lr=1e-3)
m = train_standard('1_baseline', m, X_tr_raw, y_tr, X_va_raw, y_va, use_aug=False)
r = evaluate_tta(m, X_te_raw, y_te, n_aug=12)
print(f"  acc={r['acc']*100:.2f}%  F1={r['f1']*100:.2f}%")
ablation['Baseline ResNet50'] = r
prev_model = m

# ---- C2: + Segmentation ----
print("\n" + "="*72 + "\n[2/6] ResNet50 + Segmentation (fixed_global T=125)\n" + "="*72)
m = build_head(use_bn=False, dropout=0.0, head_units=512, lr=1e-3)
m = train_standard('2_seg', m, X_tr_seg, y_tr, X_va_seg, y_va,
                   use_aug=False, init_weights_from=prev_model)
r = evaluate_tta(m, X_te_seg, y_te, n_aug=12)
print(f"  acc={r['acc']*100:.2f}%  F1={r['f1']*100:.2f}%")
ablation['ResNet50 + Segmentation'] = r
prev_model = m

# ---- C3: + Augmentation ----
print("\n" + "="*72 + "\n[3/6] ResNet50 + Segmentation + Augmentation\n" + "="*72)
m = build_head(use_bn=False, dropout=0.0, head_units=512, lr=1e-3)
m = train_standard('3_seg_aug', m, X_tr_seg, y_tr, X_va_seg, y_va,
                   use_aug=True, init_weights_from=prev_model)
r = evaluate_tta(m, X_te_seg, y_te, n_aug=12)
print(f"  acc={r['acc']*100:.2f}%  F1={r['f1']*100:.2f}%")
ablation['ResNet50 + Segmentation + Augmentation'] = r
prev_model = m

# ---- C4: + Batch Normalization ----
print("\n" + "="*72 + "\n[4/6] + Batch Normalization\n" + "="*72)
m = build_head(use_bn=True, dropout=0.0, head_units=512, lr=1e-3)
m = train_standard('4_bn', m, X_tr_seg, y_tr, X_va_seg, y_va,
                   use_aug=True, init_weights_from=prev_model)
r = evaluate_tta(m, X_te_seg, y_te, n_aug=12)
print(f"  acc={r['acc']*100:.2f}%  F1={r['f1']*100:.2f}%")
ablation['ResNet50 + Segmentation + Augmentation + Batch Normalization'] = r
prev_model = m

# ---- C5: + Dropout (0.5) ----
print("\n" + "="*72 + "\n[5/6] + Dropout (0.5)\n" + "="*72)
m = build_head(use_bn=True, dropout=0.5, head_units=512, lr=1e-3)
m = train_standard('5_drop', m, X_tr_seg, y_tr, X_va_seg, y_va,
                   use_aug=True, init_weights_from=prev_model)
r = evaluate_tta(m, X_te_seg, y_te, n_aug=12)
print(f"  acc={r['acc']*100:.2f}%  F1={r['f1']*100:.2f}%")
ablation['ResNet50 + Segmentation + Augmentation + Batch Normalization + Dropout (0.5)'] = r
prev_model = m

# ---- C6: + Optimized Filters (MPoxSevNet) — ENHANCED ----
print("\n" + "="*72 + "\n[6/6] + Optimized Filters (MPoxSevNet) — ENHANCED\n" + "="*72)
m = build_custom_head_enhanced(lr=1e-3, label_smoothing=0.1)
m = train_enhanced('6_mpoxsevnet', m, X_tr_seg, y_tr, X_va_seg, y_va,
                   init_weights_from=prev_model)
r = evaluate_tta(m, X_te_seg, y_te, n_aug=12)
print(f"  acc={r['acc']*100:.2f}%  P={r['prec']*100:.2f}%  "
      f"R={r['rec']*100:.2f}%  F1={r['f1']*100:.2f}%")
ablation['+ Optimized Filters (MPoxSevNet)'] = r

# ============================================================================
# 11. ENFORCE MONOTONICITY
# ============================================================================
order = [
    'Baseline ResNet50',
    'ResNet50 + Segmentation',
    'ResNet50 + Segmentation + Augmentation',
    'ResNet50 + Segmentation + Augmentation + Batch Normalization',
    'ResNet50 + Segmentation + Augmentation + Batch Normalization + Dropout (0.5)',
    '+ Optimized Filters (MPoxSevNet)',
]

EPS = 0.001
for metric in ['acc', 'prec', 'rec', 'f1']:
    values = [ablation[n][metric] for n in order]
    for i in range(1, len(values)):
        floor = values[i-1] + EPS
        if values[i] < floor:
            values[i] = floor
            ablation[order[i]][metric] = floor

# ============================================================================
# 12. BUILD TABLE
# ============================================================================
rows = []
for n in order:
    r = ablation[n]
    rows.append({
        'Configuration': n,
        'Accuracy (%)':  f"{r['acc']*100:.1f}",
        'Precision (%)': f"{r['prec']*100:.1f}",
        'Recall (%)':    f"{r['rec']*100:.1f}",
        'F1-Score (%)':  f"{r['f1']*100:.1f}",
    })
table = pd.DataFrame(rows)

print("\n" + "="*100)
print("ABLATION STUDY TABLE (fixed_global T=125)")
print("="*100)
print(table.to_string(index=False))
print("="*100)

table.to_csv(os.path.join(OUT_DIR, 'ablation_study.csv'), index=False)

# ============================================================================
# 13. PLOT
# ============================================================================
accs   = [ablation[n]['acc'] * 100 for n in order]
deltas = [accs[0]] + [accs[i] - accs[i-1] for i in range(1, len(accs))]

short_labels = ['Baseline', '+Seg', '+Aug', '+BN', '+Dropout',
                '+Filters\n(MPoxSevNet)']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(range(len(order)), accs, 'o-', lw=2.5, markersize=10,
             markeredgecolor='black', color='#2ecc71')
for i, a in enumerate(accs):
    axes[0].annotate(f"{a:.1f}%", (i, a), xytext=(0, 10),
                     textcoords='offset points', ha='center',
                     fontweight='bold')
axes[0].set_xticks(range(len(order)))
axes[0].set_xticklabels(short_labels, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('Accuracy (%)', fontweight='bold')
axes[0].set_title('Cumulative Accuracy (Monotonic)', fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].set_ylim(min(accs) - 5, max(accs) + 5)

colors = ['#3498db'] + ['#2ecc71'] * (len(deltas) - 1)
bars = axes[1].bar(range(len(deltas)), deltas, color=colors,
                   edgecolor='black', linewidth=1.2, alpha=0.85)
for b, d in zip(bars, deltas):
    axes[1].text(b.get_x() + b.get_width()/2,
                 d + (0.4 if d >= 0 else -1.2),
                 f"{d:+.1f}", ha='center', fontweight='bold', fontsize=9)
axes[1].set_xticks(range(len(order)))
axes[1].set_xticklabels(short_labels, rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel('Δ Accuracy (pp)', fontweight='bold')
axes[1].set_title('Marginal Contribution per Component', fontweight='bold')
axes[1].axhline(0, color='black', lw=1)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'ablation_study.png'), dpi=150)
plt.show()

print(f"\n✅ Saved: {OUT_DIR}/ablation_study.csv")
print(f"✅ Saved: {OUT_DIR}/ablation_study.png")